In [ ]:
import json
from pathlib import Path

PROJ_ROOT = Path().resolve().parent
storage_dir = PROJ_ROOT / "storage" / "test_answer_table"
# input_path = storage_dir / "20251111ccb2023plus.json"
input_path = storage_dir / "poc_bj_jdbg.json"
question_path = storage_dir / "question_task_temp.json"
output_path = storage_dir / "output.json"
import httpx

# 不用asyncio/async，改为同步方式
def run_answer_table_test_sync():
    # 读取 content_list
    with open(input_path, "r", encoding="utf-8") as f:
        content_list = json.load(f)

    # 读取 question_tasks
    with open(question_path, "r", encoding="utf-8") as f:
        question_tasks = json.load(f)
    
    results = []
    url = "http://localhost:6300/api/table_agent/answer_table"
    # 设置超时时间
    timeout = httpx.Timeout(1800.0)  # 30分钟
    # 按Streaming event-stream方式拉取，直接处理流
    with httpx.Client(timeout=timeout) as client, open(output_path, "w", encoding="utf-8") as f_out:
        body = {
            "content_list": content_list,
            "question_tasks": question_tasks
        }
        try:
            # 注意stream=True用于event-stream响应
            with client.stream("POST", url, json=body) as resp:
                if resp.status_code == 200:
                    # FastAPI 返回 "text/event-stream"
                    for line in resp.iter_lines():
                        if not line:
                            continue
                        # 每条都是以 "data: " 开头
                        if isinstance(line, bytes):
                            line = line.decode("utf-8")
                        if line.startswith("data:"):
                            payload = line.removeprefix("data:").strip()
                            if not payload:
                                continue
                            try:
                                obj = json.loads(payload)
                                # 出错统一处理
                                if isinstance(obj, dict) and ("code" in obj or "error_message" in obj):
                                    # 本轮推送是错误事件
                                    results.append(obj)
                                    f_out.seek(0)
                                    json.dump(results, f_out, ensure_ascii=False, indent=2)
                                    f_out.truncate()
                                else:
                                    # 结果事件(正常一条)
                                    results.append(obj)
                                    f_out.seek(0)
                                    json.dump(results, f_out, ensure_ascii=False, indent=2)
                                    f_out.truncate()
                            except Exception as e:
                                results.append({"error": f"解析返回数据出错: {e}", "raw": payload})
                                f_out.seek(0)
                                json.dump(results, f_out, ensure_ascii=False, indent=2)
                                f_out.truncate()
                else:
                    # 错误信息整体写入
                    result = {"error": f"HTTP {resp.status_code}", "detail": resp.text}
                    results.append(result)
                    f_out.seek(0)
                    json.dump(results, f_out, ensure_ascii=False, indent=2)
                    f_out.truncate()
        except Exception as e:
            result = {"error": str(e)}
            results.append(result)
            f_out.seek(0)
            json.dump(results, f_out, ensure_ascii=False, indent=2)
            f_out.truncate()

    return results

result = run_answer_table_test_sync()
print(result)


In [ ]:
# print(result)
is_answerable=result[0]["answer"]["is_answerable"]
citations = result[0]["answer"]["citations"]
print(citations)
if is_answerable:
    answer_text=result[0]["answer"]["answer_text"]
    from IPython.display import display, HTML

    display(HTML(answer_text))

## MCP接口的测试

In [ ]:
import os
from fastmcp import FastMCP, Client

server11 = "10.192.72.11:6300"
server12 = "10.192.72.12:6300"
localhost = "127.0.0.1:6300"
client = Client(f"http://{localhost}/api/content_generate/mcp", timeout=360000)  # 生成时间较长，需要增加timeout时间

async with client:
    ping_res = await client.ping()
    print(ping_res)
    tools = await client.list_tools()
    # 仅打印工具名，避免把完整 schema 输出到 notebook 结果中
    print([t.name for t in tools])

print("done")

In [ ]:
import json
from pathlib import Path
import asyncio, json
from typing import List, Dict

# content_file_path = Path("../storage/20251111ccb2023plus.json")
# content_file_path = Path("../storage/20251125test.json")
# content_file_path = Path("../storage/test_answer_table/20251111ccb2023plus.json")
content_file_path = Path("../storage/test_answer_table/poc_bj_jdbg.json")
with content_file_path.open("r", encoding="utf-8") as f:
    content_list = json.load(f)

# question_tasks_file_path = Path("../storage/question_tasks.json")
# question_tasks_file_path = Path("../storage/question_tasks_origin.json")
question_tasks_file_path = Path("../storage/test_answer_table/question_task_temp.json")
with question_tasks_file_path.open("r", encoding="utf-8") as f:
    question_tasks_from_file_full : List[Dict] = json.load(f)
    # print(question_tasks_from_file_full)
    question_tasks_from_file = [q for q in question_tasks_from_file_full]
    question_tasks = question_tasks_from_file

# 兼容多种形态的进度事件
async def on_progress(event, *args, **kwargs):
    print(args[1]) # message

# 输出事件 - 改进版本，增加错误处理和日志
async def print_event(event):
    try:
        print(f"[事件] {event}")
    except Exception as e:
        print(f"[错误] 处理事件时出错: {e}, 事件: {event}")

async def run(content_list, question_tasks):
    # task_list = QuestionTask()
    args = {
        "contentList": content_list,
        "questionTasks": question_tasks,
        # "model_id": "model_for_content_generation",
    }
    try:
        async with client:
            # 关键：progress_handler 用于流式消费事件；timeout 给长任务足够时间
            r = await client.call_tool(
                "content_generate_sse",
                args,
                timeout=36000,
                progress_handler=print_event
            )
            # 最终聚合结果（服务器最后一次推送的结构化结果）
            print("\n[final content]:")
            print(r)
            return r
    except asyncio.CancelledError:
        print("\n[警告] 任务被取消")
        raise
    except Exception as e:
        print(f"\n[错误] 执行任务时出错: {e}")
        import traceback
        traceback.print_exc()
        raise
        
print(len(question_tasks))

# 使用 asyncio.create_task 和 try-except 确保可以正常中断
# 这样可以更好地处理 kernel 中断信号
task = None
try:
    task = asyncio.create_task(run(content_list, question_tasks))
    result = await task
except KeyboardInterrupt:
    print("\n[中断] 收到中断信号，正在取消任务...")
    if task is not None:
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            print("[中断] 任务已成功取消")
    else:
        print("[中断] 任务尚未创建，直接退出")
except Exception as e:
    print(f"\n[错误] 执行失败: {e}")
    import traceback
    traceback.print_exc()